In [ ]:
from pyexpat.errors import messages
!pip install langchain langchain-ollama langchain-community langchain-experimental pypdf chromadb

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

print("Setup configuration complete!")

In [5]:
from langchain_ollama import ChatOllama

llama_llm = ChatOllama(
    model="glm-4.6:cloud",
    temperature=0.2
)
print("Cloud-backed DeepSeek LLM is ready!")

Cloud-backed DeepSeek LLM is ready!


In [6]:
test_llm = llama_llm.invoke("who is the mans best friend?")

In [8]:
print(test_llm.content)

That's a classic question with a classic answer!

The universally recognized answer to "Who is man's best friend?" is the **dog**.

This saying is incredibly popular because it captures a unique and long-standing bond between humans and canines. Here’s a deeper look at why the dog holds this title:

### 1. Unconditional Loyalty and Companionship
Dogs are famous for their unwavering devotion. They offer companionship without judgment, are consistently happy to see us, and provide a sense of comfort and security that is often unmatched. This non-judgmental affection is a powerful antidote to the complexities of human relationships.

### 2. A Long, Shared History
The partnership between humans and dogs dates back tens of thousands of years. They were the first animals to be domesticated, and this relationship was built on mutual benefit. Dogs helped humans with hunting, guarding, and herding, while humans provided food and shelter. This shared history is woven into the fabric of our civil

In [9]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages = [
    SystemMessage("You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
    HumanMessage("I like high-intensity workouts, what should I do?"),
    AIMessage("You should try a CrossFit class"),
    HumanMessage("How often should I attend?")
]

response = llama_llm.invoke(messages)
print(response.content)

Start with two to three sessions per week.


In [10]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

string_prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")
formatted_string = string_prompt.invoke({"adjective": "funny", "topic": "cats"})
print(formatted_string.text)

Tell me one funny joke about cats


In [11]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a helpful assistance"),
    ("user", "tell me a joke about {topic}")
])

formatted_chat = chat_prompt.invoke({"topic": "cats"})
print(formatted_chat)

messages=[SystemMessage(content='you are a helpful assistance', additional_kwargs={}, response_metadata={}), HumanMessage(content='tell me a joke about cats', additional_kwargs={}, response_metadata={})]


In [12]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

json_parser = JsonOutputParser()

format_instructions = """
RESPONSE FORMAT: Return ONLY a single JSON object—no markdown block wrappers, no conversational text. Must strictly look like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}
"""

prompt_template = PromptTemplate(
    template= """
    You are a strict JSON extraction assistance.
    Task: Generate information about the movie "{movie_name}".
    {format_instructions}
    """,
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions}
)

movie_chain = prompt_template | llama_llm | json_parser

result = movie_chain.invoke({"movie_name": "The Matrix"})

print(result["title"])
print(result["director"])
print(result["year"])
print(result["genre"])

The Matrix
The Wachowskis
1999
Science fiction


In [13]:
!pip install langchain-text-splitters


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
document = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(document)

print("successfully split the document")

successfully split the document


In [16]:
!pip install langchain-huggingface langchain-chroma sentence-transformers

   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   --------------------------------------- 588.9/588.9 kB 13.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   -------------------------- ------------- 7.1/10.8 MB 35.8 MB/s eta 0:00:01
   ---------------------------------------- 10.8/10.8 MB 33.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 31.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ------------------------------ --------- 6.0/8.0 MB 30.7 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 27.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ------- -------------------------------- 7.1/36.5 MB 33.4 MB/s eta 0:00:01
   --------------- ------------------------ 14.4/36.5 MB 34.1 MB/s eta 0:00:01
   -----------------------


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(chunks, embedding_model)

retrieve = vector_store.as_retriever(search_kwargs={"k":2})

print("Vector store and retriever are set up!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 862.95it/s]


Vector store and retriever are set up!


In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """
Use the following pieces of context to answer the question at the end.
If you do not know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}
Helpful Answer:
"""

custom_rag_prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retrieve | format_docs, "question": RunnablePassthrough()}
    | custom_rag_prompt
    | llama_llm
    | StrOutputParser()
)

query = "what is langchain?"
result = qa_chain.invoke(query)

print(query)
print(result)

what is langchain?
Based on the provided context, LangChain provides a prebuilt agent architecture and model integrations to help you get started quickly and seamlessly incorporate LLMs (Large Language Models) into your agents and applications.


In [19]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a helpful and friendly assistance"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llama_llm

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

config = {"configurable": {"session_id": "cat_session"}}

D:\GitHub\IBM-RAG-and-Agentic-AI-Professional-Certificate\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [21]:

input_text = "Hello, I am a little cat. Who are you?"
response = with_message_history.invoke(
    {"input": input_text},
    config=config
)
print(response.content)

Well, hello there, little cat! It's a pleasure to meet you.

You can just think of me as a friendly helper. I don't have soft fur or a twitchy tail like you, and I can't nap in a sunbeam (though it sounds wonderful!). Instead, I'm a friendly voice that lives in the glowing rectangles your humans are always looking at.

I'm here to answer questions, tell stories, or just chat. What's on your mind today? Are you looking for the coziest spot for a nap, or do you want to hear a story about a very brave mouse?


In [23]:

input_text = "who am i again?"
response = with_message_history.invoke(
    {"input": input_text},
    config=config
)
print(response.content)


You are a little cat!

A very curious and clever little cat, with soft fur, twitchy whiskers, and a tail that probably has a mind of its own. You're an expert at napping, pouncing, and getting exactly what you want from your humans with just a look.

Don't you worry, I haven't forgotten. It's a pleasure to chat with you.


In [24]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

sentiment_template = "Analyze the sentiment of this review as positive, negative, or neutral. Review: {review}\nSentiment:"
response_template = "Write a short, professional response to the customer based on their review ({review}) and its sentiment ({sentiment})"

sentiment_prompt = PromptTemplate.from_template(sentiment_template)
response_prompt = PromptTemplate.from_template(response_template)


sentiment_step = sentiment_prompt | llama_llm | StrOutputParser()
response_step = response_prompt | llama_llm | StrOutputParser()

overall_chain = (
    {"review": RunnablePassthrough()}
    | RunnablePassthrough.assign(sentiment=sentiment_step)
    | RunnablePassthrough.assign(response=response_step)
)

review_text = "I absolutely love this laptop! It is so fast and the screen is beautiful, but the battery life is terrible."

result = overall_chain.invoke(review_text)

print("Original Review:", result['review'])
print("\nDetected Sentiment:", result['sentiment'])
print("\nCustomer Service Response:\n", result['response'])

Original Review: I absolutely love this laptop! It is so fast and the screen is beautiful, but the battery life is terrible.

Detected Sentiment: Neutral

Customer Service Response:
 Thank you for your feedback. We're so glad you're loving the laptop's speed and beautiful screen!

We're sorry to hear the battery life has not met your expectations. Our support team would be happy to help you with some optimization tips or look into this further. You can reach them at [support email/phone].


In [26]:
!pip install langgraph


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def calculator(expression: str) -> str:
    """Useful for performing simple math calculations. Input should be a mathematical expression like '2 + 2'."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"


tools = [calculator]

print("tool created successfully")

tool created successfully


In [28]:
agent = create_agent(
    model=llama_llm,
    tools=tools,
    system_prompt="You are a helpful mathematical assistant. Always use the calculator tool for math questions."
)
print("agent assembled")

agent assembled


In [30]:
inputs = {
    "messages": [
        {"role": "user", "content": "What is 250 multiplied by 4, and then divided by 2?"}
    ]
}

print("input ready")

input ready


In [34]:
print("Executing Agent...\n")

for chunk in agent.stream(inputs):
    # 1. Check if this chunk came from the model or the tools
    root_key = "model" if "model" in chunk else "tools" if "tools" in chunk else None

    if root_key:
        # 2. Safely grab the newest message inside that dictionary layer
        last_message = chunk[root_key]["messages"][-1]

        # SCENARIO A: The AI decides it needs to call our calculator tool
        if last_message.type == "ai" and last_message.tool_calls:
            tool_call = last_message.tool_calls[0]
            print(f"Agent Action: Decided to use '{tool_call['name']}' with formula {tool_call['args']}")

        # SCENARIO B: The tool finished running and reported back the calculation
        elif last_message.type == "tool":
            print(f"Observation: The calculator output was {last_message.content}")

        # SCENARIO C: The AI reads the final result and explains it to you
        elif last_message.type == "ai" and last_message.content:
            print(f"\nFinal Answer: {last_message.content}")

print("\nTask Complete!")

Executing Agent...

Agent Action: Decided to use 'calculator' with formula {'expression': '250 * 4 / 2'}
Observation: The calculator output was 500.0

Final Answer: 250 multiplied by 4, and then divided by 2 equals **500**.

Task Complete!
